In [1]:
import os
import pandas as pd
import numpy as np
import re
import sys
from scipy.stats import wilcoxon
from statistics import mean

# ==============================================================================
# CONFIGURACIÓN DE RUTAS Y PARÁMETROS
# ==============================================================================
datasets = ["braaksc", "ceradsc", "cogdx"]
k_values = [25, 50, 75, 100, 500, 1000, 1500, 2500]
results_root = "../results"
output_dir = "./k_value_comparison"
metrics = ["BA", "F1", "PS"]

PARSIMONY_SUMMARY_CSV = "../best_k_selection/parsimony_analysis/parsimony_summary.csv"
OUTPUT_LOG_FILE = os.path.join(output_dir, "k_champions_league_results.txt")

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Clase simple para duplicar la salida de print a la consola y al archivo TXT
class Logger(object):
    def __init__(self, filename):
        self.terminal = sys.stdout
        self.log = open(filename, "w", encoding="utf-8")

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)

    def flush(self):
        self.terminal.flush()
        self.log.flush()

# Redirigir la salida estándar
sys.stdout = Logger(OUTPUT_LOG_FILE)

# ==============================================================================
# FUNCIONES ESTADÍSTICAS Y DE RANKING
# ==============================================================================

def calculate_net_wins(scores, method="signed_rank"):
    """
    Calcula un sistema de victorias netas entre diferentes configuraciones de K.
    
    Parámetros:
    - scores: DataFrame con los resultados de las métricas.
    - method: 'signed_rank' utiliza el test de Wilcoxon (p < 0.05) para validar 
      diferencias significativas. 'mean' realiza comparación directa de promedios.
    """
    labels = scores.columns.values
    n_candidates = len(labels)
    net_wins = {label: 0 for label in labels}
    
    for i in range(n_candidates - 1):
        for j in range(i + 1, n_candidates):
            col_i = scores.iloc[:, i].values
            col_j = scores.iloc[:, j].values
            
            winner = None
            
            if method == "mean":
                if mean(col_i) > mean(col_j): winner = i
                elif mean(col_j) > mean(col_i): winner = j
            
            elif method == "signed_rank":
                try:
                    if not np.array_equal(col_i, col_j):
                        _, p_value = wilcoxon(col_i, col_j)
                        if p_value < 0.05:
                            if mean(col_i) > mean(col_j): winner = i
                            else: winner = j
                except:
                    pass 
            
            if winner == i:
                net_wins[labels[i]] += 1
                net_wins[labels[j]] -= 1
            elif winner == j:
                net_wins[labels[j]] += 1
                net_wins[labels[i]] -= 1
                
    return net_wins

# ==============================================================================
# GESTIÓN DE DATOS Y LECTURA DE ARCHIVOS
# ==============================================================================

def get_winner_info(dataset, k):
    """
    Extrae el nombre del algoritmo ganador de un experimento previo mediante regex.
    """
    py_path = os.path.join(results_root, dataset, f"k_{k}", "best_analysis", f"{dataset}_best_dataset_k{k}.py")
    if not os.path.exists(py_path): 
        return None
    
    try:
        with open(py_path, 'r', encoding='utf-8') as f:
            content = f.read()
        match = re.search(r"metodologia_ganador\s*=\s*['\"](.+?)['\"]", content)
        return match.group(1) if match else None
    except:
        return None

def load_metric_data(dataset, k, winner_method):
    """
    Carga los vectores de resultados (folds) para las métricas seleccionadas.
    Determina la ruta del archivo CSV basándose en el prefijo de la metodología (R1, R2, R3).
    """
    base_name = f"genes-{dataset}"
    
    if "R1" in winner_method:
        suffix = f"_{base_name}_k{k}.csv"
    elif "R2" in winner_method:
        suffix = f"_{base_name}-FR-{k}.csv"
    elif "R3" in winner_method:
        suffix = f"_{base_name}-resampling_FR-{k}.csv"
    else:
        suffix = f"_{base_name}_k{k}.csv"

    clean_method = winner_method.split("-")[0]
    metrics_data = {}
    
    for metric in metrics:
        csv_path = os.path.join(results_root, dataset, f"k_{k}", f"test_{metric}{suffix}")
        
        if os.path.exists(csv_path):
            df = pd.read_csv(csv_path)
            col = next((c for c in [winner_method, clean_method] if c in df.columns), None)
            
            if col:
                metrics_data[metric] = df[col].values[:10]
            else:
                metrics_data[metric] = None
        else:
            metrics_data[metric] = None
            
    return metrics_data.get("BA"), metrics_data.get("F1"), metrics_data.get("PS")

# ==============================================================================
# PROCESO PRINCIPAL: LIGA DE CAMPEONES K
# ==============================================================================

df_global_parsimony = None
if os.path.exists(PARSIMONY_SUMMARY_CSV):
    df_global_parsimony = pd.read_csv(PARSIMONY_SUMMARY_CSV)

for dataset in datasets:
    print(f"\n{'='*80}")
    print(f" LIGA DE CAMPEONES K: {dataset.upper()}")
    print(f"{'='*80}")
    
    df_ba, df_f1, df_ps = pd.DataFrame(), pd.DataFrame(), pd.DataFrame()
    k_methodologies = {}
    
    for k in k_values:
        winner_name = get_winner_info(dataset, k)
        if not winner_name: continue
        
        col_name = f"K={k}"
        k_methodologies[col_name] = winner_name
        data_ba, data_f1, data_ps = load_metric_data(dataset, k, winner_name)
        
        if data_ba is not None and data_f1 is not None and data_ps is not None:
            df_ba[col_name], df_f1[col_name], df_ps[col_name] = data_ba, data_f1, data_ps

    if df_ba.empty:
        print("     Datos insuficientes para procesar este dataset.")
        continue

    sr_ba = calculate_net_wins(df_ba, "signed_rank")
    sr_f1 = calculate_net_wins(df_f1, "signed_rank")
    sr_ps = calculate_net_wins(df_ps, "signed_rank")
    
    mean_ba = calculate_net_wins(df_ba, "mean")
    mean_f1 = calculate_net_wins(df_f1, "mean")
    mean_ps = calculate_net_wins(df_ps, "mean")

    results = []
    for col in df_ba.columns:
        total_sr = sr_ba[col] + sr_f1[col] + sr_ps[col]
        total_mean = mean_ba[col] + mean_f1[col] + mean_ps[col]
        results.append({
            "K_Option": col,
            "Methodology": k_methodologies.get(col, "N/A"),
            "Total_Signed_Rank": total_sr,
            "Total_Mean_Rank": total_mean,
            "Details_SR": f"BA:{sr_ba[col]} F1:{sr_f1[col]} PS:{sr_ps[col]}"
        })

    df_res = pd.DataFrame(results).sort_values(by=["Total_Signed_Rank", "Total_Mean_Rank"], ascending=False)
    
    print(f"\n RANKING FINAL DE COMPETICIÓN")
    print("-" * 105)
    print(df_res[["K_Option", "Methodology", "Total_Signed_Rank", "Total_Mean_Rank", "Details_SR"]].to_string(index=False))
    print("-" * 105)
    
    best_k_stat = df_res.iloc[0]["K_Option"]
    best_method_stat = df_res.iloc[0]["Methodology"]
    print(f" GANADOR ESTADÍSTICO PARA {dataset.upper()}: {best_k_stat} ({best_method_stat})")

    if df_global_parsimony is not None:
        row_pars = df_global_parsimony[df_global_parsimony['Dataset'] == dataset]
        if not row_pars.empty:
            chosen_k_val = row_pars.iloc[0]['K_elegido']
            k_label_pars = f"K={chosen_k_val}"
            
            match_pars = df_res[df_res['K_Option'] == k_label_pars]
            if not match_pars.empty:
                perf = match_pars.iloc[0]
                print(f" GANADOR POR PARSIMONIA PARA {dataset.upper()}: {k_label_pars} ({perf['Methodology']})")
                print(f"   └─ Comparativa en Liga: SR={perf['Total_Signed_Rank']} | Mean={perf['Total_Mean_Rank']} | {perf['Details_SR']}")
            else:
                print(f" GANADOR POR PARSIMONIA PARA {dataset.upper()}: {k_label_pars} (No presente en ranking)")
    else:
        print(" [INFO] No se encontró el archivo de parsimonia para realizar la comparativa.")

print(f"\n{'='*80}")
print(f" PROCESO FINALIZADO")
print(f" Report saved to: {OUTPUT_LOG_FILE}")
print(f"{'='*80}")


 LIGA DE CAMPEONES K: BRAAKSC

 RANKING FINAL DE COMPETICIÓN
---------------------------------------------------------------------------------------------------------
K_Option              Methodology  Total_Signed_Rank  Total_Mean_Rank        Details_SR
    K=75             ROS_RF-R2_FR                  9               13   BA:6 F1:5 PS:-2
    K=50 BorderlineSMOTE_RF-R2_FR                  5                7   BA:2 F1:5 PS:-2
  K=1000          AllKNN_RF-R2_FR                  5                7   BA:1 F1:-1 PS:5
   K=100          AllKNN_RF-R2_FR                  3                5   BA:1 F1:-2 PS:4
  K=1500          AllKNN_RF-R2_FR                  1               -1  BA:-1 F1:-3 PS:5
    K=25             ROS_RF-R2_FR                 -2               -3  BA:-1 F1:3 PS:-4
  K=2500           Tomek_RF-R2_FR                -10              -13  BA:-4 F1:-6 PS:0
   K=500     KMeansSMOTE_RF-R2_FR                -11              -15 BA:-4 F1:-1 PS:-6
----------------------------------------